# 02_credit_risk_default_prediction — Modeling & Evaluation

Train a production-style sklearn pipeline, evaluate on holdout data, and export artifacts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
import joblib

                ROOT = Path('..').resolve()
                df = pd.read_csv(ROOT / 'data' / 'credit_risk.csv')
                target = 'defaulted'

                X = df.drop(columns=[target])
                y = df[target]

                id_cols = [c for c in X.columns if c.endswith('_id') or c in ['customer_id','application_id','transaction_id']]
                X_model = X.drop(columns=id_cols)

                cat_cols = [c for c in X_model.columns if X_model[c].dtype == 'object']
                num_cols = [c for c in X_model.columns if c not in cat_cols]

                pre = ColumnTransformer([
                    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
                    ('num', StandardScaler(), num_cols)
                ])

                # Load the trained model artifact from /models, if present; otherwise retrain using run_pipeline.py.
                model_path = ROOT / 'models' / 'credit_risk_gb.joblib'
                print('Model file:', model_path)


## Train/Test split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X_model, y, test_size=0.2, random_state=42, stratify=y)


## Train a baseline (logistic regression) for interpretability

In [ ]:

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1500, class_weight='balanced' if y.mean() < 0.2 else None)
pipe = Pipeline([('prep', pre), ('model', clf)])
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:,1]
pred = (proba >= 0.5).astype(int)

metrics = {
    'roc_auc': float(roc_auc_score(y_test, proba)),
    'accuracy': float(accuracy_score(y_test, pred)),
    'precision': float(precision_score(y_test, pred, zero_division=0)),
    'recall': float(recall_score(y_test, pred, zero_division=0)),
    'f1': float(f1_score(y_test, pred, zero_division=0))
}
metrics


## Plots (ROC / PR / Confusion Matrix)

In [ ]:

RocCurveDisplay.from_predictions(y_test, proba)
plt.title("ROC Curve")
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, proba)
plt.title("Precision-Recall Curve")
plt.show()

cm = confusion_matrix(y_test, pred)
ConfusionMatrixDisplay(cm).plot(values_format='d')
plt.title("Confusion Matrix (thr=0.50)")
plt.show()


## Explainability (Top coefficients)
For a linear model, coefficients provide a direct view of drivers.

In [ ]:

# Extract feature names and coefficients
ohe = pipe.named_steps['prep'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(cat_cols)) if len(cat_cols) else []
feat_names = cat_names + list(num_cols)

coefs = pipe.named_steps['model'].coef_.ravel()
coef_df = pd.DataFrame({'feature': feat_names, 'coef': coefs}).sort_values('coef', ascending=False)
top_pos = coef_df.head(12)
top_neg = coef_df.tail(12).sort_values('coef')
top_pos, top_neg


## Save retrained artifacts

In [ ]:

(ROOT / 'models').mkdir(exist_ok=True)
(ROOT / 'reports').mkdir(exist_ok=True)

joblib.dump(pipe, ROOT / 'models' / 'baseline_logreg_notebook.joblib')
Path(ROOT / 'reports' / 'baseline_metrics_notebook.json').write_text(json.dumps(metrics, indent=2))
coef_df.to_csv(ROOT / 'reports' / 'baseline_coefficients.csv', index=False)
print("Saved baseline model + metrics + coefficients.")
